In [2]:
print("Test")

Test


In [3]:
import pandas as pd
import evaluate

In [4]:
ud_test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")

In [5]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")


def custom_metrics(preds, labels):

    # Evaluate using poseval
    result = poseval.compute(predictions=preds, references=labels)

    return result

In [6]:
# 1) Load libs + helpers
import pandas as pd
import torch
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import classification_report
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# import helper functions from your preprocessing module
from scripts.model.model import (
    initialize_model,
    _group_sentences_from_df,
    prepare_token_classification_data,
    TokenClassificationDataset,
)

In [7]:
# 2) Read test data and keep required cols
test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")
test_df = test_df[["sentence_id", "words", "labels"]]

In [8]:
# 3) Load model+tokenizer (pass model folder or checkpoint)
model_bundle = initialize_model(
    "../models/NER_mudel_v2_homonym/"
)  # returns model, tokenizer, id2label, label2id
model = model_bundle["model"]
tokenizer = model_bundle["tokenizer"]
id2label = model_bundle["id2label"]
label2id = model_bundle["label2id"]

E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\model.py:112: UserWarning: Using label mapping from model `id2label` in checkpoint.
  warnings.warn("Using label mapping from model `id2label` in checkpoint.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
# 4) Group into sentence-level (list of (words_list, labels_list))
sentences = _group_sentences_from_df(test_df)

In [10]:
# 5) Build encodings & aligned labels (use same max_length you trained with)
encodings, aligned_labels = prepare_token_classification_data(
    tokenizer,
    sentences,
    label2id,
    max_length=None,  # adjust if you used different length
    ignore_placeholders=False,  # choose True if you want to ignore non-target placeholders
)

In [12]:
for i, (sent, label) in enumerate(sentences):
    if len(sent) < 30:
        print(i)
        print(sent)
        break

240
['Kuid', 'siis', 'nad', 'tulid', ',', 'elusignaalid', '.', 'Lindistus', '!', 'Kas', 'sari', 'on', 'tänaseks', 'põhjalikult', 'läbi', 'võetud', '?', 'Raha', 'on', 'hea', 'asi', '.', 'Järgnevalt', 'käsitletaksegi', 'nimetatud', 'teguriterühmi', 'lähemalt', '.']


In [13]:
# confirm fast tokenizer
print("is_fast:", tokenizer.is_fast)

# tokenise single sentence and inspect BatchEncoding
enc = tokenizer(sentences[i][0], is_split_into_words=True, return_offsets_mapping=True)
# print(enc)  # full encoding
print(
    "tokens:", enc.tokens(), "length:", len(enc.tokens())
)  # token strings (includes special tokens)
print(
    "input_ids:", enc["input_ids"], "length:", len(enc["input_ids"])
)  # token ids (includes special tokens)
print(
    "attention_mask:", enc["attention_mask"], "length:", len(enc["attention_mask"])
)  # attention mask (includes special tokens)
print(
    "word_ids:", enc.word_ids(), "length:", len(enc.word_ids())
)  # word ids for each token (None for special tokens)
print(
    "offsets:", enc["offset_mapping"], "length:", len(enc["offset_mapping"])
)  # character offsets for each token (includes special tokens)
print(
    "labels:", sentences[i][1], "length:", len(sentences[i][1])
)  # original labels for the sentence
print(
    "labels input format:", aligned_labels[i], "length:", len(aligned_labels[i])
)  # labels aligned to tokens (with -100 for ignored tokens)

is_fast: True
tokens: ['<s>', '▁Ku', 'id', '▁s', 'iis', '▁n', 'ad', '▁t', 'ul', 'id', '▁,', '▁el', 'us', 'ig', 'na', 'al', 'id', '▁.', '▁L', 'in', 'di', 'st', 'us', '▁!', '▁K', 'as', '▁sa', 'ri', '▁on', '▁tä', 'na', 'se', 'ks', '▁põh', 'ja', 'li', 'ku', 'lt', '▁lä', 'bi', '▁võ', 'et', 'ud', '▁?', '▁Ra', 'ha', '▁on', '▁h', 'ea', '▁a', 'si', '▁.', '▁Järg', 'ne', 'va', 'lt', '▁kä', 'sit', 'le', 'ta', 'kse', 'gi', '▁n', 'im', 'et', 'at', 'ud', '▁te', 'gu', 'ri', 'ter', 'üh', 'mi', '▁lä', 'he', 'ma', 'lt', '▁.', '</s>'] length: 79
input_ids: [5, 222, 20, 21, 125, 32, 92, 11, 48, 20, 36, 313, 30, 173, 58, 12, 20, 42, 137, 13, 141, 9, 30, 513, 65, 67, 163, 41, 45, 145, 58, 7, 34, 319, 17, 91, 110, 124, 214, 243, 72, 94, 26, 414, 600, 111, 45, 61, 368, 27, 71, 42, 3479, 81, 35, 124, 205, 895, 10, 25, 178, 63, 32, 59, 94, 82, 26, 50, 85, 41, 452, 464, 250, 214, 107, 22, 124, 42, 6] length: 79
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,